In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

In [ ]:
"""
TVT 预测 —— 特征工程主脚本
输入：横井 CSV + 标准井 CSV
输出：model_train.csv / model_test.csv（每行 = 1ft 横井深度点，带完整特征）

特征体系：
  A. 轨迹几何特征       —— 井斜、方位、狗腿度、曲率
  B. 地层距离特征        —— 距各地质分界面的距离（仅训练集有地层列时可用）
  C. GR 原始 & 滤波      —— 多尺度平滑、导数、包络
  D. GR 统计窗口特征     —— 滑动均值/std/min/max/偏度/峰度
  E. TVT_input 特征      —— 已知段的 TVT 趋势外推
  F. GR-标准井对比特征   —— 互相关深度偏移、DTW 距离、最近邻 GR 残差
  G. 地质标签编码特征    —— 标准井对比推断出的软地层归属概率
  H. 空间位置特征        —— X/Y 归一化、沿侧钻方向累积距离
"""

import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter1d
from scipy.signal import correlate
from scipy.stats import skew, kurtosis
from scipy.interpolate import interp1d
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# ──────────────────────────────────────────────
# 0. 配置
# ──────────────────────────────────────────────
TRAIN_DIR  = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction/train")
TEST_DIR   = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction/test")
OUT_DIR    = Path("/kaggle/working/")
OUT_DIR.mkdir(parents=True, exist_ok=True)

FORMATIONS = ["ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA"]
TARGET_COL = "TVT"
SEED       = 42

# GR 滑动窗口尺寸（ft）
WINDOWS = [5, 11, 21, 51, 101]

# 标准井 GR 互相关搜索范围（ft）
CC_SEARCH  = 100
CC_WIN     = 50        # 每次取多少 ft 的窗口做互相关


# ──────────────────────────────────────────────
# 1. 轨迹几何特征
# ──────────────────────────────────────────────
def trajectory_features(hw: pd.DataFrame) -> pd.DataFrame:
    """
    井斜角、方位角、狗腿度、曲率、侧钻进入标志
    利用 X/Y/Z 差分反算几何量
    """
    md  = hw["MD"].values
    x   = hw["X"].values
    y   = hw["Y"].values
    z   = hw["Z"].values
    dmd = np.diff(md, prepend=md[0])
    dmd = np.where(dmd == 0, 1e-6, dmd)   # 防零除

    dx = np.diff(x, prepend=x[0])
    dy = np.diff(y, prepend=y[0])
    dz = np.diff(z, prepend=z[0])

    horiz_dist = np.sqrt(dx**2 + dy**2)

    # 井斜角 (inclination)，0=垂直，90=水平
    inc = np.degrees(np.arctan2(horiz_dist, dz))
    inc = np.clip(inc, 0, 90)

    # 方位角 (azimuth)
    az  = np.degrees(np.arctan2(dx, dy)) % 360

    # 狗腿度 (dogleg severity, deg/100ft)
    inc_diff = np.diff(inc, prepend=inc[0])
    az_diff  = np.diff(az,  prepend=az[0])
    az_diff  = (az_diff + 180) % 360 - 180        # wrap to [-180,180]
    dogleg   = np.sqrt(inc_diff**2 + az_diff**2) / dmd * 100

    # 累积水平位移（沿侧钻方向的距离，用于空间趋势）
    cum_horiz = np.cumsum(horiz_dist)

    # 侧钻标志：inc >= 80° 认为进入水平段
    is_lateral = (inc >= 80).astype(float)

    # 距侧钻起点的 MD 距离
    lateral_start_md = md[inc >= 80][0] if (inc >= 80).any() else md[-1]
    md_from_lateral  = np.clip(md - lateral_start_md, 0, None)

    feat = pd.DataFrame({
        "inc"             : np.round(inc, 3),
        "az"              : np.round(az,  3),
        "dogleg"          : np.round(dogleg, 4),
        "dz_per_dmd"      : np.round(dz / dmd, 5),   # 垂深变化率
        "horiz_dist"      : np.round(horiz_dist, 3),
        "cum_horiz"       : np.round(cum_horiz, 2),
        "is_lateral"      : is_lateral,
        "md_from_lateral" : np.round(md_from_lateral, 1),
        # 平滑后的井斜（减少噪声）
        "inc_smooth5"     : np.round(gaussian_filter1d(inc, 5), 3),
    })
    return feat


# ──────────────────────────────────────────────
# 2. 地层深度辅助模型 + 地层距离特征
# ──────────────────────────────────────────────

class FormationPredictor:
    """
    用训练集已知的地层深度列（ANCC/ASTNU/...），
    训练 6 个轻量回归模型，输入为每个深度点的
    空间坐标+测量深度+GR，输出为各地层顶面深度。

    训练集和测试集统一用此模型预测地层深度，
    确保特征空间完全一致，消除训练/测试不对称。
    """

    # 用于预测地层深度的输入特征（训练/测试集都有）
    INPUT_COLS = ["MD", "X", "Y", "Z", "GR"]

    def __init__(self):
        self.models_     = {}    # {formation: sklearn model}
        self.is_fitted_  = False

    def save(self, path: str) -> None:
        """保存整个 FormationPredictor 到文件（joblib 序列化）"""
        import joblib
        joblib.dump(self, path)
        print(f"  [FormationPredictor] 已保存 → {path}")

    @classmethod
    def load(cls, path: str) -> "FormationPredictor":
        """从文件加载已训练好的 FormationPredictor"""
        import joblib
        predictor = joblib.load(path)
        print(f"  [FormationPredictor] 已加载 ← {path}")
        print(f"  包含地层模型: {list(predictor.models_.keys())}")
        return predictor

    def fit(self, hw_df_list: list,
             max_samples: int = 500_000) -> "FormationPredictor":
        """
        hw_df_list  : list of pd.DataFrame，每个是一口训练井的横井数据
        max_samples : 最大训练样本数，随机下采样防止500万行超时
                      设为 None 则使用全量数据
        """
        # HistGradientBoostingRegressor：原生支持 NaN，大数据集速度快
        from sklearn.ensemble import HistGradientBoostingRegressor

        # 汇总所有训练井数据（只保留有地层列的井）
        all_rows = pd.concat(
            [df for df in hw_df_list
             if all(f in df.columns for f in FORMATIONS)],
            ignore_index=True
        )
        if len(all_rows) == 0:
            raise ValueError("没有可用的训练井（缺少地层列）")

        # ★ 删除 X 或 y 中含 NaN 的行
        cols_needed = self.INPUT_COLS + FORMATIONS
        n_before    = len(all_rows)
        all_rows    = all_rows[cols_needed].dropna()
        n_dropped   = n_before - len(all_rows)
        if n_dropped > 0:
            print(f"  [去NaN] 删除含NaN行 {n_dropped:,} 行，剩余 {len(all_rows):,} 行")

        total = len(all_rows)
        # ★ 随机下采样：500万行没必要全用，50万行已足够学到空间规律
        if max_samples and total > max_samples:
            all_rows = all_rows.sample(n=max_samples, random_state=SEED)
            print(f"  [下采样] {total:,} → {max_samples:,} 行")

        X_aux = all_rows[self.INPUT_COLS].values

        print(f"\n[FormationPredictor] 训练辅助模型")
        print(f"  训练样本数: {len(X_aux):,}  输入特征: {self.INPUT_COLS}")

        for form in FORMATIONS:
            y_f = all_rows[form].values
            mdl = HistGradientBoostingRegressor(
                max_iter=200,           # 等价于 n_estimators
                learning_rate=0.05,
                max_depth=6,
                min_samples_leaf=50,    # 大数据防过拟合
                random_state=SEED,
            )
            mdl.fit(X_aux, y_f)        # ★ NaN 直接支持，无需预处理
            r2 = mdl.score(X_aux, y_f)
            self.models_[form] = mdl
            print(f"  {form:<8}: 训练 R²={r2:.4f}")

        self.is_fitted_ = True
        return self

    def predict(self, hw: pd.DataFrame) -> dict:
        """
        入力：单口井的横井 DataFrame（只需含 INPUT_COLS）
        出力：{formation: np.array(预测地层深度)} 
        """
        if not self.is_fitted_:
            raise RuntimeError("请先调用 fit()")
        X = hw[self.INPUT_COLS].values
        return {f: self.models_[f].predict(X) for f in FORMATIONS}


def formation_features(hw: pd.DataFrame,
                        formation_pred: dict) -> pd.DataFrame:
    """
    用预测的地层深度计算 dist_* 和 norm_pos 特征。

    formation_pred : FormationPredictor.predict() 的输出
                     {formation: np.array(预测深度)}
    训练集和测试集统一走此路径，特征含义完全一致。
    """
    z    = hw["Z"].values
    feat = {}

    for f in FORMATIONS:
        pred_depth    = formation_pred[f]
        feat[f"dist_{f}"] = np.round(z - pred_depth, 3)

    # 归一化层间位置：在 EGFDU ~ BUDA 之间的相对位置 [0,1]
    top    = formation_pred["EGFDU"]
    bottom = formation_pred["BUDA"]
    thick  = bottom - top
    thick  = np.where(thick == 0, 1e-6, thick)
    feat["norm_pos_EGFDU_BUDA"] = np.round((z - top) / thick, 4)

    return pd.DataFrame(feat, index=hw.index)



# ──────────────────────────────────────────────
# 3. GR 多尺度特征
# ──────────────────────────────────────────────
def gr_features(hw: pd.DataFrame) -> pd.DataFrame:
    gr = hw["GR"].values.copy()
    feat = {}

    # 原始
    feat["GR_raw"] = gr

    # 多尺度高斯平滑
    for w in WINDOWS:
        sg = int(w / 3) or 1
        feat[f"GR_smooth_{w}"] = np.round(gaussian_filter1d(gr, sg), 3)

    # 一阶导数（变化率）
    feat["GR_d1"]       = np.round(np.gradient(gr), 4)
    feat["GR_d1_abs"]   = np.round(np.abs(np.gradient(gr)), 4)
    # 二阶导数（曲率）
    feat["GR_d2"]       = np.round(np.gradient(np.gradient(gr)), 5)

    # 高频成分（原始 - 低频）
    feat["GR_hf"] = np.round(gr - gaussian_filter1d(gr, 20), 3)

    # 滑动统计窗口
    s = pd.Series(gr)
    for w in [11, 21, 51]:
        r = s.rolling(w, center=True, min_periods=1)
        feat[f"GR_mean_{w}"]  = np.round(r.mean().values, 3)
        feat[f"GR_std_{w}"]   = np.round(r.std().values, 3)
        feat[f"GR_min_{w}"]   = np.round(r.min().values, 3)
        feat[f"GR_max_{w}"]   = np.round(r.max().values, 3)
        feat[f"GR_range_{w}"] = np.round(
            r.max().values - r.min().values, 3)
        feat[f"GR_skew_{w}"]  = np.round(
            s.rolling(w, center=True, min_periods=3).skew().values, 4)

    # GR 相对于全局均值/std 的 z-score
    gmu = np.nanmean(gr)
    gsd = np.nanstd(gr) + 1e-6
    feat["GR_zscore"] = np.round((gr - gmu) / gsd, 4)

    return pd.DataFrame(feat, index=hw.index)


# ──────────────────────────────────────────────
# 4. TVT_input 趋势特征
# ──────────────────────────────────────────────
def tvt_input_features(hw: pd.DataFrame) -> pd.DataFrame:
    """
    利用已知段的 TVT_input 构建趋势特征：
    - 距已知段末尾的 MD 距离
    - 已知段末尾 TVT 值（锚点）
    - 已知段末尾 TVT 的局部斜率（趋势）
    - 线性外推 TVT 值（地质学假设：层厚缓变）
    """
    ti   = hw["TVT_input"].values.copy()
    md   = hw["MD"].values
    feat = {}

    known_mask = ~np.isnan(ti)
    known_idx  = np.where(known_mask)[0]

    if len(known_idx) == 0:
        feat["tvt_anchor"]       = np.nan
        feat["tvt_local_slope"]  = np.nan
        feat["tvt_extrap"]       = np.nan
        feat["md_from_known_end"]= np.nan
        feat["tvt_input_filled"] = np.nan
        return pd.DataFrame(feat, index=hw.index)

    last_known = known_idx[-1]
    anchor_tvt = ti[last_known]
    anchor_md  = md[last_known]

    # 局部斜率：用末尾 50 ft 做线性拟合
    win50 = known_idx[known_idx >= last_known - 50]
    if len(win50) >= 2:
        slope = np.polyfit(md[win50], ti[win50], 1)[0]
    else:
        slope = 0.0

    # 线性外推
    tvt_extrap = anchor_tvt + slope * (md - anchor_md)

    # 用外推值填充 NaN（仅特征，不作为 label）
    ti_filled = ti.copy()
    ti_filled[~known_mask] = tvt_extrap[~known_mask]

    feat["tvt_anchor"]        = np.round(anchor_tvt, 3)
    feat["tvt_local_slope"]   = np.round(slope, 6)
    feat["tvt_extrap"]        = np.round(tvt_extrap, 3)
    feat["md_from_known_end"] = np.round(md - anchor_md, 2)
    feat["tvt_input_filled"]  = np.round(ti_filled, 3)
    feat["tvt_input_raw"]     = ti          # 原始（含 NaN）

    # 滑动平均（对已知段平滑）
    ts = pd.Series(ti)
    feat["tvt_input_smooth11"] = np.round(
        ts.rolling(11, center=True, min_periods=1).mean().values, 3)

    return pd.DataFrame(feat, index=hw.index)


# ──────────────────────────────────────────────
# 5. GR–标准井对比特征（地质统计核心）
# ──────────────────────────────────────────────
def typewell_correlation_features(hw: pd.DataFrame,
                                   tw: pd.DataFrame) -> pd.DataFrame:
    """
    将横井 GR 与标准井 GR 做相关分析，估计横井当前的 TVT 位置。

    方法：
    (a) 最近邻 TVT 匹配：找标准井中 GR 最近似的深度点
    (b) 滑动窗口互相关：估计局部 TVT 偏移
    (c) GR 残差：横井 GR - 标准井对应位置 GR
    (d) 软地层归属概率：基于 GR 与每层均值的距离
    """
    hw_gr  = hw["GR"].values
    hw_md  = hw["MD"].values
    tw_tvt = tw["TVT"].values
    tw_gr  = tw["GR"].values

    # ★ 修复：测试集标准井没有 Geology 列，用 GR 分位数自动推断软标签
    if "Geology" in tw.columns:
        tw_geo = tw["Geology"].values
    else:
        # 按 GR 值从低到高将标准井深度段分配到 6 个地层
        # （GR 低 → 致密层如 BUDA，GR 高 → 泥岩如 ANCC）
        geo_labels = ["ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA"]
        quantiles  = np.linspace(0, 1, len(geo_labels) + 1)
        bounds     = np.quantile(tw_gr, quantiles)
        tw_geo     = np.full(len(tw_gr), "ABOVE", dtype=object)
        for idx, g in enumerate(geo_labels):
            lo, hi = bounds[idx], bounds[idx + 1]
            mask   = (tw_gr >= lo) & (tw_gr <= hi if idx == len(geo_labels)-1 else tw_gr < hi)
            tw_geo[mask] = g

    n = len(hw_gr)
    feat = {}

    # ── (a) 最近邻 GR → 估计 TVT ──────────────────────────────
    # 对每个横井点，在标准井中找 GR 差值最小的点
    nn_tvt      = np.zeros(n)
    nn_gr_resid = np.zeros(n)

    for i in range(n):
        diff  = np.abs(tw_gr - hw_gr[i])
        j     = np.argmin(diff)
        nn_tvt[i]      = tw_tvt[j]
        nn_gr_resid[i] = hw_gr[i] - tw_gr[j]

    feat["nn_tvt_est"]   = np.round(nn_tvt,      3)
    feat["nn_gr_resid"]  = np.round(nn_gr_resid, 3)

    # 平滑版本（减少噪声）
    feat["nn_tvt_smooth11"] = np.round(
        gaussian_filter1d(nn_tvt, 5), 3)

    # ── (b) 滑动窗口互相关 → 局部 TVT 偏移 ───────────────────
    # 以 TVT_input 已知段末尾为锚点，做 CC_WIN ft 窗口的互相关
    ti_vals       = hw["TVT_input"].values
    known_mask    = ~np.isnan(ti_vals)
    known_idx     = np.where(known_mask)[0]
    cc_tvt_offset = np.zeros(n)

    if len(known_idx) > CC_WIN:
        anchor_idx = known_idx[-1]
        anchor_tvt = ti_vals[anchor_idx]

        # 从锚点向后，每 CC_WIN ft 做一次互相关
        for i in range(n):
            win_start = max(0, i - CC_WIN // 2)
            win_end   = min(n, win_start + CC_WIN)
            hw_win    = hw_gr[win_start:win_end]

            # 标准井中对应 TVT 附近的窗口
            tvt_center = anchor_tvt + (hw_md[i] - hw_md[anchor_idx]) * 0.0
            tw_center_idx = np.searchsorted(tw_tvt, tvt_center)
            tw_start  = max(0, tw_center_idx - CC_WIN - CC_SEARCH)
            tw_end    = min(len(tw_gr), tw_center_idx + CC_WIN + CC_SEARCH)
            tw_win    = tw_gr[tw_start:tw_end]

            if len(hw_win) < 5 or len(tw_win) < len(hw_win):
                continue

            cc = correlate(tw_win - tw_win.mean(),
                           hw_win - hw_win.mean(), mode="valid")
            offset_idx = np.argmax(cc)
            # 偏移量（ft）
            cc_tvt_offset[i] = (offset_idx - CC_SEARCH) * 1.0

    feat["cc_tvt_offset"] = np.round(cc_tvt_offset, 2)

    # ── (c) 标准井插值 GR（按 TVT_input 对齐）─────────────────
    # 用 TVT_input（已知段）在标准井插值，获得"期望 GR"
    tw_gr_interp_fn = interp1d(tw_tvt, tw_gr,
                                bounds_error=False, fill_value="extrapolate")

    ti_filled = feat.get("tvt_input_filled",
                          np.where(known_mask, ti_vals, nn_tvt))
    # 若 tvt_input_filled 尚未计算，用 nn_tvt 代替
    if isinstance(ti_filled, dict):
        ti_filled = nn_tvt

    # 标准井对应 GR
    tw_gr_at_tvt = tw_gr_interp_fn(ti_vals)
    feat["tw_gr_at_tvt_input"] = np.round(tw_gr_at_tvt, 3)
    feat["gr_vs_tw_resid"]     = np.round(hw_gr - tw_gr_at_tvt, 3)

    # ── (d) 软地层归属概率（基于 GR）────────────────────────
    # 每层 GR 均值（从标准井统计）
    geo_labels = ["ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA"]
    layer_gr_mean = {}
    layer_gr_std  = {}
    for g in geo_labels:
        mask = tw_geo == g
        if mask.sum() > 0:
            layer_gr_mean[g] = tw_gr[mask].mean()
            layer_gr_std[g]  = tw_gr[mask].std() + 1e-3
        else:
            layer_gr_mean[g] = 50.0
            layer_gr_std[g]  = 10.0

    # 高斯似然 → softmax 概率
    log_probs = np.zeros((n, len(geo_labels)))
    for j, g in enumerate(geo_labels):
        log_probs[:, j] = -0.5 * ((hw_gr - layer_gr_mean[g]) /
                                    layer_gr_std[g])**2

    log_probs -= log_probs.max(axis=1, keepdims=True)
    probs = np.exp(log_probs)
    probs /= probs.sum(axis=1, keepdims=True)

    for j, g in enumerate(geo_labels):
        feat[f"prob_{g}"] = np.round(probs[:, j], 4)

    # 最可能地层的 GR 均值
    best_layer_idx = np.argmax(probs, axis=1)
    feat["best_layer_gr_mean"] = np.array(
        [layer_gr_mean[geo_labels[k]] for k in best_layer_idx])

    # ── (e) 标准井 GR 统计特征（全局参考）───────────────────
    feat["tw_gr_mean"]   = float(tw_gr.mean())
    feat["tw_gr_std"]    = float(tw_gr.std())
    feat["tw_tvt_range"] = float(tw_tvt.max() - tw_tvt.min())

    return pd.DataFrame(feat, index=hw.index)


# ──────────────────────────────────────────────
# 6. GR_TVT 特征：标准井 TVT → GR 映射到横井
# ──────────────────────────────────────────────
def gr_tvt_features(hw: pd.DataFrame, tw: pd.DataFrame) -> pd.DataFrame:
    """
    核心思路：
      标准井以 TVT 为深度轴记录了 GR 曲线（GR vs TVT）。
      横井的 TVT_input（已知段）提供了每个深度点的 TVT 值。
      将两者结合，可以为横井每个点生成：
        GR_TVT        —— 标准井在该 TVT 处的 GR 插值（"期望GR"）
        GR_TVT_diff   —— 横井实测 GR 与期望 GR 的差值（地质偏差）
        GR_TVT_ratio  —— 横井 GR / 期望 GR（相对偏差）
      以及多尺度平滑、导数、滑动统计版本。

    对 TVT_input 为 NaN 的评估区（需要预测的区域），
    用已知段末尾的线性外推 TVT 先填充，再做插值。
    """
    tw_tvt = tw["TVT"].values
    tw_gr  = tw["GR"].values
    hw_gr  = hw["GR"].values
    ti     = hw["TVT_input"].values.copy().astype(float)
    md     = hw["MD"].values
    n      = len(hw_gr)

    # ── Step1：填充 TVT_input 的 NaN（评估区）用线性外推 ──────
    known_mask = ~np.isnan(ti)
    known_idx  = np.where(known_mask)[0]

    ti_filled = ti.copy()
    if len(known_idx) >= 2:
        last   = known_idx[-1]
        win50  = known_idx[known_idx >= last - 50]
        slope  = np.polyfit(md[win50], ti[win50], 1)[0] if len(win50) >= 2 else 0.0
        anchor = ti[last]
        for i in range(last + 1, n):
            ti_filled[i] = anchor + slope * (md[i] - md[last])
    elif len(known_idx) == 1:
        ti_filled[np.isnan(ti_filled)] = ti[known_idx[0]]
    # 若完全无已知值，ti_filled 仍为 NaN，后续插值结果也为 NaN

    # ── Step2：建立标准井 TVT→GR 插值函数 ────────────────────
    # 标准井按 TVT 排序（防乱序）
    sort_idx  = np.argsort(tw_tvt)
    tw_tvt_s  = tw_tvt[sort_idx]
    tw_gr_s   = tw_gr[sort_idx]

    gr_interp_fn = interp1d(tw_tvt_s, tw_gr_s,
                             kind="linear",
                             bounds_error=False,
                             fill_value=(tw_gr_s[0], tw_gr_s[-1]))  # 边界外用端值

    # ── Step3：为横井每个点生成 GR_TVT ───────────────────────
    gr_tvt_vals = gr_interp_fn(ti_filled)          # 标准井期望 GR

    gr_tvt_diff  = hw_gr - gr_tvt_vals             # 实测 - 期望
    gr_tvt_ratio = hw_gr / np.where(gr_tvt_vals == 0, 1e-6, gr_tvt_vals)

    feat = {
        "GR_TVT"       : np.round(gr_tvt_vals,  3),   # 标准井对应 TVT 处的 GR
        "GR_TVT_diff"  : np.round(gr_tvt_diff,  3),   # 横井GR - 标准井GR
        "GR_TVT_ratio" : np.round(gr_tvt_ratio, 4),   # 横井GR / 标准井GR
    }

    # ── Step4：多尺度平滑版本 ─────────────────────────────────
    for sigma in [3, 7, 15]:
        feat[f"GR_TVT_smooth{sigma}"] = np.round(
            gaussian_filter1d(gr_tvt_vals, sigma), 3)
        feat[f"GR_TVT_diff_smooth{sigma}"] = np.round(
            gaussian_filter1d(gr_tvt_diff, sigma), 3)

    # ── Step5：一阶导数（标准井GR随TVT的变化率）─────────────
    feat["GR_TVT_d1"]     = np.round(np.gradient(gr_tvt_vals), 4)
    feat["GR_TVT_d1_abs"] = np.round(np.abs(np.gradient(gr_tvt_vals)), 4)
    feat["GR_TVT_diff_d1"]= np.round(np.gradient(gr_tvt_diff), 4)

    # ── Step6：滑动统计窗口（在 TVT 轴上的局部统计）─────────
    s_tvt  = pd.Series(gr_tvt_vals)
    s_diff = pd.Series(gr_tvt_diff)
    for w in [11, 21, 51]:
        r = s_tvt.rolling(w, center=True, min_periods=1)
        feat[f"GR_TVT_mean_{w}"]  = np.round(r.mean().values, 3)
        feat[f"GR_TVT_std_{w}"]   = np.round(r.std().values,  3)
        feat[f"GR_TVT_range_{w}"] = np.round(
            (r.max() - r.min()).values, 3)
        feat[f"GR_TVT_diff_mean_{w}"] = np.round(
            s_diff.rolling(w, center=True, min_periods=1).mean().values, 3)

    # ── Step7：标准井 GR 在 TVT 轴的全局统计（常量，作为参考基准）
    feat["tw_gr_at_tvt_global_mean"] = float(np.nanmean(gr_tvt_vals))
    feat["tw_gr_at_tvt_global_std"]  = float(np.nanstd(gr_tvt_vals))

    return pd.DataFrame(feat, index=hw.index)


# ──────────────────────────────────────────────
# 7. 空间位置特征
# ──────────────────────────────────────────────
def spatial_features(hw: pd.DataFrame) -> pd.DataFrame:
    x = hw["X"].values
    y = hw["Y"].values
    z = hw["Z"].values

    # 归一化到 [0,1]（相对井内位置）
    def norm01(v):
        mn, mx = v.min(), v.max()
        return (v - mn) / (mx - mn + 1e-9)

    feat = pd.DataFrame({
        "x_norm"       : np.round(norm01(x), 5),
        "y_norm"       : np.round(norm01(y), 5),
        "z_norm"       : np.round(norm01(z), 5),
        "md_norm"      : np.round(norm01(hw["MD"].values), 5),
        # 相对于全井 X/Y 起点的偏移
        "x_offset"     : np.round(x - x[0], 2),
        "y_offset"     : np.round(y - y[0], 2),
        # Z 相对于 Z 中位数的偏差（判断是否抬升/下沉）
        "z_vs_median"  : np.round(z - np.median(z), 3),
    })
    return feat


# ──────────────────────────────────────────────
# 7. 主流程：单口井特征构建
# ──────────────────────────────────────────────
def build_well_features(hw: pd.DataFrame,
                         tw: pd.DataFrame,
                         well_name: str,
                         formation_pred: dict) -> pd.DataFrame:
    # ── 基础列：TVT / TVT_input 原样保留，不做任何变更 ──
    # 排除原始文件里可能已有的 WELLNAME 列，统一从文件名重新赋值
    keep_cols = ["MD", "X", "Y", "Z", "GR"]
    if "TVT"       in hw.columns: keep_cols.append("TVT")
    if "TVT_input" in hw.columns: keep_cols.append("TVT_input")
    base = hw[keep_cols].copy()
    # ★ WELLNAME 从文件名提取，插入为第一列
    base.insert(0, "WELLNAME", well_name)

    # 各组特征
    traj   = trajectory_features(hw)
    form   = formation_features(hw, formation_pred)   # ★ 传入预测地层深度
    gr_f   = gr_features(hw)
    tvt_f  = tvt_input_features(hw)
    tw_f   = typewell_correlation_features(hw, tw)
    gr_tvt = gr_tvt_features(hw, tw)
    sp_f   = spatial_features(hw)

    df = pd.concat([base   .reset_index(drop=True),
                    traj   .reset_index(drop=True),
                    form   .reset_index(drop=True),
                    gr_f   .reset_index(drop=True),
                    tvt_f  .reset_index(drop=True),
                    tw_f   .reset_index(drop=True),
                    gr_tvt .reset_index(drop=True),
                    sp_f   .reset_index(drop=True)],
                   axis=1)
    return df


# ──────────────────────────────────────────────
# 8. 批量处理所有井
# ──────────────────────────────────────────────
def process_directory(data_dir: Path,
                       predictor: FormationPredictor,
                       is_train: bool = True) -> pd.DataFrame:
    hw_files = sorted(data_dir.glob("*__horizontal_well.csv"))
    all_dfs  = []

    tag = "训练" if is_train else "测试"
    print(f"\n{'='*55}")
    print(f"处理{tag}集：{len(hw_files)} 口井")
    print('='*55)

    for hw_path in hw_files:
        # ★ 从文件名 {WELLNAME}__horizontal_well.csv 提取 WELLNAME
        well_name = hw_path.stem.replace("__horizontal_well", "")
        tw_path   = data_dir / f"{well_name}__typewell.csv"

        if not tw_path.exists():
            print(f"  [跳过] {well_name}：找不到标准井文件")
            continue

        hw = pd.read_csv(hw_path)
        tw = pd.read_csv(tw_path)

        # ★ 统一用辅助模型预测地层深度（训练/测试集一致）
        formation_pred = predictor.predict(hw)

        df = build_well_features(hw, tw, well_name, formation_pred)
        all_dfs.append(df)

        n_total = len(df)
        n_eval  = df["TVT"].isna().sum() if is_train else \
                  df["TVT_input"].isna().sum()
        print(f"  {well_name}  rows={n_total:>6,}  "
              f"eval_zone={n_eval:>5,} ({n_eval/n_total:.0%})  "
              f"features={df.shape[1]}")

    combined = pd.concat(all_dfs, ignore_index=True)
    return combined


# ──────────────────────────────────────────────
# 9. 运行 & 保存
# ──────────────────────────────────────────────
def main():
    # ══════════════════════════════════════════
    # フェーズ1：辅助模型训练
    # 只读取训练集横井，拿地层列训练 FormationPredictor
    # ══════════════════════════════════════════
    print("\n" + "="*55)
    print("Phase 1：训练地层深度辅助模型")
    print("="*55)

    PREDICTOR_PATH = OUT_DIR / "formation_predictor.joblib"

    if PREDICTOR_PATH.exists():
        # ★ 已有模型直接加载，跳过重新训练
        print(f"  [跳过训练] 检测到已保存的模型：{PREDICTOR_PATH}")
        predictor = FormationPredictor.load(str(PREDICTOR_PATH))
    else:
        # 首次运行：读取训练井数据，训练并保存
        train_hw_files = sorted(TRAIN_DIR.glob("*__horizontal_well.csv"))
        train_hw_list  = [pd.read_csv(p) for p in train_hw_files]

        predictor = FormationPredictor()
        predictor.fit(train_hw_list)

        # ★ 训练完立即保存
        predictor.save(str(PREDICTOR_PATH))

    # ══════════════════════════════════════════
    # フェーズ2：特征工程（训练集 + 测试集统一用辅助模型）
    # ══════════════════════════════════════════
    print("\n" + "="*55)
    print("Phase 2：特征工程")
    print("="*55)

    train_df = process_directory(TRAIN_DIR, predictor, is_train=True)

    train_labeled   = train_df[train_df["TVT"].notna()].copy()
    train_eval_zone = train_df[train_df["TVT"].isna()].copy()

    test_df = process_directory(TEST_DIR, predictor, is_train=False)

    # ── 保存 ──
    train_df.to_csv(OUT_DIR / "features_train_full.csv",  index=False)
    train_labeled.to_csv(OUT_DIR / "features_train_labeled.csv", index=False)
    train_eval_zone.to_csv(OUT_DIR / "features_train_eval_zone.csv", index=False)
    test_df.to_csv(OUT_DIR / "features_test.csv",  index=False)

    # ── 报告 ──
    print(f"\n{'='*55}")
    print("特征工程完成")
    print(f"{'='*55}")
    print(f"训练集（全量）  : {train_df.shape[0]:>8,} 行 × {train_df.shape[1]} 列")
    print(f"训练集（有标签）: {train_labeled.shape[0]:>8,} 行  ← 用于模型训练")
    print(f"训练集（评估区）: {train_eval_zone.shape[0]:>8,} 行  ← 验证集")
    print(f"测试集          : {test_df.shape[0]:>8,} 行")
    print(f"\n输出目录: {OUT_DIR}")
    print(f"  features_train_labeled.csv   ← 模型训练输入")
    print(f"  features_train_eval_zone.csv ← 验证/评估")
    print(f"  features_test.csv            ← 预测提交")

    # ── 特征列表 ──
    feature_cols = [c for c in train_labeled.columns
                    if c not in ["WELLNAME", "TVT", "TVT_input",
                                 "tvt_input_raw", "GR", "MD"]]
    print(f"\n特征总数: {len(feature_cols)}")
    groups = {
        "轨迹几何": [c for c in feature_cols if any(
            c.startswith(p) for p in ["inc","az","dogleg","dz_","horiz","cum_horiz","is_lat","md_from_lat"])],
        "地层距离": [c for c in feature_cols if c.startswith("dist_") or "norm_pos" in c or "n_form" in c],
        "GR多尺度": [c for c in feature_cols if c.startswith("GR_")],
        "TVT趋势" : [c for c in feature_cols if c.startswith("tvt_")],
        "标准井对比":[c for c in feature_cols if any(
            c.startswith(p) for p in ["nn_","cc_","tw_","gr_vs","prob_","best_layer"])],
        "空间位置": [c for c in feature_cols if any(
            c.endswith(s) for s in ["_norm","_offset","vs_median"])],
    }
    for gname, gcols in groups.items():
        print(f"\n  【{gname}】 ({len(gcols)} 个)")
        for c in gcols:
            print(f"    {c}")

    # ── 简单统计 ──
    print(f"\n目标变量 TVT 统计（有标签行）：")
    print(train_labeled["TVT"].describe().round(3).to_string())

    return train_labeled, test_df


if __name__ == "__main__":
    train_labeled, test_df = main()